In [1]:
import os
import wave
import tempfile
import numpy as np
import sounddevice as sd
from scipy.io.wavfile import write
from dotenv import load_dotenv

In [2]:
# ─────────────────────────────────────────
# Configuration
# ─────────────────────────────────────────
load_dotenv()

SAMPLE_RATE     = 16000   # Hz — optimal for speech recognition
RECORD_SECONDS  = 10      # Max recording duration
CHANNELS        = 1       # Mono audio

DEPARTMENTS = {
    "Emergency Department": {
        "description": "Handles medical emergencies, fires, accidents, and life-threatening situations.",
        "contact": "Emergency Hotline: 911 / Internal Ext: 100",
        "priority": "CRITICAL"
    },
    "Security Department": {
        "description": "Handles theft, trespassing, suspicious activity, and access control issues.",
        "contact": "Security Control Room: Internal Ext: 200",
        "priority": "HIGH"
    }
}

In [3]:
# ─────────────────────────────────────────
# Step 1: Record audio from microphone
# ─────────────────────────────────────────
def record_audio(duration: int = RECORD_SECONDS) -> str:
    """Records audio from the microphone and saves it as a temporary WAV file."""
    print(f"\n🎙️  Recording for {duration} seconds... Speak now!")
    audio_data = sd.rec(
        int(duration * SAMPLE_RATE),
        samplerate=SAMPLE_RATE,
        channels=CHANNELS,
        dtype="int16"
    )
    sd.wait()  # Wait until recording is complete
    print("✅  Recording complete.")

    # Save to a temporary WAV file
    tmp = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
    write(tmp.name, SAMPLE_RATE, audio_data)
    return tmp.name

In [4]:
# ─────────────────────────────────────────
# Step 2: Transcribe audio using Whisper
# ─────────────────────────────────────────

def transcribe_audio(audio_path: str) -> str:
    """Transcribes audio using Whisper locally."""

    import whisper

    print("\n📝  Transcribing audio with Whisper...")

    # استخدمي small لتحسين العربية
    model = whisper.load_model("small")

    result = model.transcribe(
        audio_path,
        language="ar"
    )

    transcript = result["text"].strip()

    print(f"\n📄  Transcript:\n    \"{transcript}\"")

    return transcript


In [5]:
# ─────────────────────────────────────────
# Step 3: Extract structured info locally
# ─────────────────────────────────────────

def extract_info(transcript: str) -> dict:
    """Extract problem, need, and location from text."""

    text = transcript.lower()

    problem_keywords = [
        "حريق",
        "حادث",
        "نزيف",
        "سرقة",
        "شجار"
    ]

    need_keywords = [
        "أحتاج",
        "مساعدة",
        "إسعاف",
        "شرطة"
    ]

    location_keywords = [
        "في",
        "داخل",
        "بجوار",
        "عند"
    ]

    problem = "Unknown"
    need = "Unknown"
    location = "Unknown"

    # Detect problem
    for word in problem_keywords:
        if word in text:
            problem = word
            break

    # Detect need
    for word in need_keywords:
        if word in text:
            need = "Help requested"
            break

    # Detect location (simple extraction)
    words = transcript.split()

    for i, word in enumerate(words):
        if word in location_keywords and i + 1 < len(words):
            location = words[i + 1]
            break

    return {
        "Problem": problem,
        "Need": need,
        "Location": location
    }

In [6]:
# ─────────────────────────────────────────
# Step 4: Classify locally
# ─────────────────────────────────────────

def classify_department(transcript: str) -> str:
    """Classifies text into Emergency or Security."""

    text = transcript.lower()

    emergency_keywords = [
        "حريق",
        "حادث",
        "نزيف",
        "مريض",
        "إسعاف",
        "طوارئ",
        "طارئة"
    ]

    security_keywords = [
        "سرقة",
        "شجار",
        "مشتبه",
        "دخول غير مصرح",
        "سرق"
    ]

    for word in emergency_keywords:
        if word in text:
            return "Emergency Department"

    for word in security_keywords:
        if word in text:
            return "Security Department"

    return "Emergency Department"


In [7]:
# ─────────────────────────────────────────
# Step 5: Assign to department and display result
# ─────────────────────────────────────────
def assign_department(classification: str, info: dict) -> None:
    """Displays the assignment result and department contact information."""
    dept = DEPARTMENTS.get(classification, {})

    print("\n" + "═" * 55)
    print("         INCIDENT REPORT — DEPARTMENT ASSIGNMENT")
    print("═" * 55)
    print(f"  🏢  Assigned to : {classification}")
    print(f"  ⚠️   Priority     : {dept.get('priority', 'N/A')}")
    print(f"  📋  Description  : {dept.get('description', '')}")
    print(f"  📞  Contact      : {dept.get('contact', '')}")
    print("─" * 55)
    print("  EXTRACTED INCIDENT DETAILS:")
    for key, value in info.items():
        print(f"    {key:<12}: {value}")
    print("═" * 55)

In [8]:
# ─────────────────────────────────────────
# Main pipeline
# ─────────────────────────────────────────
def run_pipeline(audio_path: str = None) -> None:
    """
    Runs the full voice analysis pipeline.
    Pass an audio_path to process an existing file,
    or leave None to record from the microphone.
    """
    try:
        # Step 1: Get audio
        if audio_path is None:
            audio_path = record_audio()
        else:
            print(f"\n📂  Using uploaded file: {audio_path}")

        # Step 2: Transcribe
        transcript = transcribe_audio(audio_path)
        if not transcript:
            print("❌  No speech detected. Please try again.")
            return

        # Step 3: Extract structured info
        info = extract_info(transcript)

        # Step 4: Classify
        classification = classify_department(transcript)

        # Step 5: Assign and display
        assign_department(classification, info)

    except FileNotFoundError:
        print(f"❌  Audio file not found: {audio_path}")
    except Exception as e:
        print(f"❌  Error: {e}")

In [9]:
# ─────────────────────────────────────────
# Entry point
# ─────────────────────────────────────────
if __name__ == "__main__":
    print("╔══════════════════════════════════════════╗")
    print("║     Voice Message Analysis System        ║")
    print("╚══════════════════════════════════════════╝")
    print("\nOptions:")
    print("  [1] Record from microphone")
    print("  [2] Use an existing audio file")

    choice = input("\nEnter choice (1 or 2): ").strip()

    if choice == "1":
        run_pipeline()                           # Records from mic
    elif choice == "2":
        path = input("Enter path to audio file: ").strip()
        run_pipeline(audio_path=path)            # Uses uploaded file
    else:
        print("Invalid choice.")

╔══════════════════════════════════════════╗
║     Voice Message Analysis System        ║
╚══════════════════════════════════════════╝

Options:
  [1] Record from microphone
  [2] Use an existing audio file

🎙️  Recording for 10 seconds... Speak now!
✅  Recording complete.

📝  Transcribing audio with Whisper...


100%|███████████████████████████████████████| 461M/461M [03:31<00:00, 2.28MiB/s]
/Users/dani/Desktop/ai audio/venv/lib/python3.11/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")



📄  Transcript:
    "السلام عليكم وسمي دانيا في حريق في المبنى احتاج مساعدة منكم أسرع وقت أنا في الربية بالرياض"

═══════════════════════════════════════════════════════
         INCIDENT REPORT — DEPARTMENT ASSIGNMENT
═══════════════════════════════════════════════════════
  🏢  Assigned to : Emergency Department
  ⚠️   Priority     : CRITICAL
  📋  Description  : Handles medical emergencies, fires, accidents, and life-threatening situations.
  📞  Contact      : Emergency Hotline: 911 / Internal Ext: 100
───────────────────────────────────────────────────────
  EXTRACTED INCIDENT DETAILS:
    Problem     : حريق
    Need        : Help requested
    Location    : حريق
═══════════════════════════════════════════════════════
